In [12]:
import sys

if 'facenet_pytorch' not in sys.modules:
  !pip install facenet-pytorch

import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from facenet_pytorch import InceptionResnetV1, fixed_image_standardization
from PIL import Image
import numpy as np
import random
import os
from sklearn.model_selection import train_test_split

In [2]:
# --- CONFIGURAZIONE ---
TRAIN_DATA_DIR = "./drive/MyDrive/BBA_Dataset_training"
VAL_SPLIT_PCT = 0.25  # Esempio: Ora voglio il 25% di validazione
BATCH_SIZE = 8
TRAIN_CHEKPOINTS = "./drive/MyDrive/train_BBA/checkpoints"
EPOCHS = 50  # Aumentiamo le epoche per permettere alla rete di vedere tutte le combinazioni
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
P = 0.7

In [14]:
class HybridTripletDataset(Dataset):
    def __init__(self, root_dir=None, samples=None, transform=None, p=0.7, is_validation=False):
        """
        Args:
            root_dir (string): Directory con tutte le immagini (usato se samples è None).
            samples (list): Lista di tuple (path, label) da usare (per creare subset).
            is_validation (bool): Se True, riduce la randomicità (es. data augmentation) per metriche stabili.
        """
        self.p = p
        self.is_validation = is_validation

        # Se passiamo dei samples specifici (post-split), usiamo quelli.
        # Altrimenti carichiamo tutto dalla cartella.
        if samples is not None:
            self.samples = samples
            # Dobbiamo dedurre le classi dai samples forniti
            self.classes = list(set([label for _, label in samples]))
        else:
            self.image_folder = datasets.ImageFolder(root_dir)
            self.samples = self.image_folder.samples
            self.classes = self.image_folder.classes

        # Mappa Label -> Lista Immagini (costruita SOLO sui samples visibili a questo dataset)
        self.label_to_imgs = {}
        for img_path, label in self.samples:
            if label not in self.label_to_imgs:
                self.label_to_imgs[label] = []
            self.label_to_imgs[label].append(img_path)

        # Rimuoviamo classi che potrebbero essere rimaste vuote o con 0 immagini dopo lo split
        self.classes = [c for c in self.classes if c in self.label_to_imgs and len(self.label_to_imgs[c]) > 0]

        # --- TRASFORMAZIONI ---
        # Training: Flip casuale
        self.transform_train = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        # Validation: Nessun flip, deterministico
        self.transform_val = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        # Augmentation forte (solo per Training)
        self.transform_augmented = transforms.Compose([
            transforms.Resize((160, 160)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        anchor_path, anchor_label = self.samples[index]

        # Selezione transform base
        base_transform = self.transform_val if self.is_validation else self.transform_train

        # --- POSITIVE ---
        all_imgs_of_subject = self.label_to_imgs[anchor_label]

        use_real_pair = False
        if len(all_imgs_of_subject) > 1:
            # In validation cerchiamo di usare sempre coppie reali se possibile per misurare la vera performance
            if self.is_validation:
                use_real_pair = True
            elif random.random() < self.p:
                use_real_pair = True

        if use_real_pair:
            # Real Positive
            pos_path = anchor_path
            while pos_path == anchor_path:
                pos_path = random.choice(all_imgs_of_subject)
            positive_img = self._load_image(pos_path, base_transform)
        else:
            # Synthetic Positive (fallback o scelta random in train)
            # In validation, se siamo costretti a usare augmentation (solo 1 foto), usiamo comunque augmentation
            # ma magari meno aggressiva? Per ora lasciamo augmented standard.
            pos_path = anchor_path
            positive_img = self._load_image(pos_path, self.transform_augmented)

        # --- NEGATIVE ---
        neg_label = anchor_label
        # Sicurezza per evitare loop infiniti se c'è 1 sola classe
        if len(self.classes) > 1:
            while neg_label == anchor_label:
                neg_label = random.choice(self.classes)

        if neg_label in self.label_to_imgs:
            neg_path = random.choice(self.label_to_imgs[neg_label])
        else:
            # Fallback estremo
            neg_path = anchor_path

        anchor_img = self._load_image(anchor_path, base_transform)
        negative_img = self._load_image(neg_path, base_transform)

        return anchor_img, positive_img, negative_img

    def _load_image(self, path, transform_pipeline):
        try:
            img = Image.open(path).convert('RGB')
            return transform_pipeline(img)
        except Exception as e:
            return torch.randn(3, 160, 160)

In [9]:
# --- CLASSE CHECKPOINT MANAGER AVANZATA ---
class CheckpointManager:
    def __init__(self, save_dir='checkpoints', model_name='best_facenet_hybrid.pth'):
        self.save_dir = save_dir
        self.file_path = os.path.join(save_dir, model_name)
        self.best_loss = float('inf')

        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)

    def save_checkpoint(self, model, optimizer, epoch, current_loss):
        """
        Salva un dizionario completo solo se la loss migliora.
        """
        if current_loss < self.best_loss:
            print(f"   >>> Record Loss! ({self.best_loss:.4f} -> {current_loss:.4f}). Salvataggio checkpoint completo...")
            self.best_loss = current_loss

            # Creiamo il dizionario con tutto ciò che serve per riprendere
            checkpoint = {
                'epoch': epoch + 1,             # Salviamo la prossima epoca da fare
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_loss': self.best_loss
            }
            torch.save(checkpoint, self.file_path)
        else:
            print(f"   >>> Nessun miglioramento (Best: {self.best_loss:.4f}).")

    def load_resume(self, model, optimizer):
        """
        Controlla se esiste un checkpoint, lo carica e restituisce l'epoca di partenza.
        """
        if os.path.isfile(self.file_path):
            print(f"--- Trovato checkpoint: {self.file_path} ---")
            checkpoint = torch.load(self.file_path)

            # 1. Carica pesi modello
            model.load_state_dict(checkpoint['model_state_dict'])

            # 2. Carica stato ottimizzatore (momentum, learning rate interni, etc.)
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

            # 3. Recupera info training
            start_epoch = checkpoint['epoch']
            self.best_loss = checkpoint['best_loss']

            print(f"--- Riprendo dall'epoca {start_epoch} con Best Loss: {self.best_loss:.4f} ---")
            return start_epoch
        else:
            print("--- Nessun checkpoint trovato. Inizio training da zero. ---")
            return 0

In [8]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.model_selection import train_test_split
# from tuomodulo import HybridTripletDataset, CheckpointManager

class Trainer:
    def __init__(self,
                 model,
                 optimizer,
                 data_dir,
                 checkpoint_manager,
                 device,
                 val_split=0.2,
                 batch_size=8,
                 epochs=10,
                 p_aug=0.7):
        """
        Trainer flessibile che accetta modello e optimizer dall'esterno.

        Args:
            model (nn.Module): Il modello PyTorch già configurato.
            optimizer (torch.optim.Optimizer): L'ottimizzatore già configurato.
            data_dir (str): Percorso del dataset.
            checkpoint_manager (obj): Gestore dei salvataggi.
            device (torch.device): Cuda o Cpu.
            val_split (float): Percentuale di dati da usare per la validazione (0.0 - 1.0).
            ...
        """
        self.model = model
        self.optimizer = optimizer
        self.data_dir = data_dir
        self.manager = checkpoint_manager
        self.device = device
        self.val_split = val_split  # <--- Parametro richiesto
        self.batch_size = batch_size
        self.epochs_to_add = epochs
        self.p_aug = p_aug

        # La Loss per ora la teniamo qui, ma potresti passare anche questa dall'esterno
        self.criterion = nn.TripletMarginLoss(margin=0.5, p=2)

        print(f"--- Inizializzazione Trainer (Val Split: {self.val_split*100}%) ---")
        self._prepare_data()

        # Assicuriamoci che il modello sia sul device corretto
        self.model.to(self.device)

    def _prepare_data(self):
        """
        Carica dati e applica lo split variabile.
        """
        print(f"1. Caricamento dati da: {self.data_dir}")
        full_dataset_struct = datasets.ImageFolder(self.data_dir)
        all_samples = full_dataset_struct.samples

        labels = [s[1] for s in all_samples]

        # Utilizzo della variabile self.val_split
        train_samples, val_samples = train_test_split(
            all_samples,
            test_size=self.val_split,
            shuffle=True,
            stratify=labels,
            random_state=42
        )

        print(f"   Split {int((1-self.val_split)*100)}/{int(self.val_split*100)} -> Train: {len(train_samples)}, Val: {len(val_samples)}")

        # Creazione Dataset
        self.train_ds = HybridTripletDataset(samples=train_samples, p=self.p_aug, is_validation=False)
        self.val_ds = HybridTripletDataset(samples=val_samples, p=self.p_aug, is_validation=True)

        # Creazione DataLoader
        self.train_loader = DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=2)
        self.val_loader = DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=2)

    def _validate_step(self):
        self.model.eval()
        running_loss = 0.0
        correct_triplets = 0
        total_triplets = 0

        with torch.no_grad():
            for anchor, positive, negative in self.val_loader:
                anchor, positive, negative = anchor.to(self.device), positive.to(self.device), negative.to(self.device)

                a_emb = self.model(anchor)
                p_emb = self.model(positive)
                n_emb = self.model(negative)

                loss = self.criterion(a_emb, p_emb, n_emb)
                running_loss += loss.item()

                a_emb = F.normalize(a_emb, p=2, dim=1)
                p_emb = F.normalize(p_emb, p=2, dim=1)
                n_emb = F.normalize(n_emb, p=2, dim=1)

                dist_ap = torch.norm(a_emb - p_emb, p=2, dim=1)
                dist_an = torch.norm(a_emb - n_emb, p=2, dim=1)

                correct_triplets += (dist_ap < dist_an).sum().item()
                total_triplets += anchor.size(0)

        avg_loss = running_loss / len(self.val_loader)
        accuracy = correct_triplets / total_triplets if total_triplets > 0 else 0
        return avg_loss, accuracy

    def fit(self):
        # Carica checkpoint (pesi e stato optimizer)
        start_epoch = self.manager.load_resume(self.model, self.optimizer)
        end_epoch = start_epoch + self.epochs_to_add

        print("-" * 60)
        print(f"Inizio Training Session: Epoca {start_epoch+1} -> {end_epoch}")
        print("-" * 60)

        for epoch in range(start_epoch, end_epoch):
            # --- TRAIN ---
            self.model.train()
            train_loss = 0.0

            for anchor, positive, negative in self.train_loader:
                anchor, positive, negative = anchor.to(self.device), positive.to(self.device), negative.to(self.device)

                self.optimizer.zero_grad()

                a_emb = self.model(anchor)
                p_emb = self.model(positive)
                n_emb = self.model(negative)

                loss = self.criterion(a_emb, p_emb, n_emb)
                loss.backward()
                self.optimizer.step()

                train_loss += loss.item()

            avg_train_loss = train_loss / len(self.train_loader)

            # --- VALIDATION ---
            avg_val_loss, val_acc = self._validate_step()

            # --- LOGGING ---
            print(f"Epoch [{epoch+1}/{end_epoch}]")
            print(f"   Train Loss: {avg_train_loss:.4f}")
            print(f"   Val Loss:   {avg_val_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%")

            # --- SAVE ---
            self.manager.save_checkpoint(self.model, self.optimizer, epoch, avg_val_loss)
            print("-" * 60)

        print("Training completato.")

In [17]:
if __name__ == "__main__":
    # 2. Configurazione Modello (Esterno al Trainer)
    print("Configurazione Modello...")
    model = InceptionResnetV1(pretrained='vggface2', classify=False).to(DEVICE)

    # --- Logica di Freezing (Esterna al Trainer) ---
    for param in model.parameters():
        param.requires_grad = False
    for param in model.block8.parameters():
        param.requires_grad = True
    for param in model.last_linear.parameters():
        param.requires_grad = True
    for param in model.last_bn.parameters():
        param.requires_grad = True

    # 3. Configurazione Optimizer (Esterno al Trainer)
    # Nota: passiamo solo i parametri con requires_grad=True
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE
    )

    # 4. Istanza Checkpoint Manager
    ckpt_manager = CheckpointManager(save_dir=TRAIN_CHEKPOINTS, model_name='BBA_best_fext.pth')

    # 5. Istanza Trainer (Iniezione delle dipendenze)
    trainer = Trainer(
        model=model,                 # <--- Passo il modello
        optimizer=optimizer,         # <--- Passo l'optimizer
        data_dir=TRAIN_DATA_DIR,
        checkpoint_manager=ckpt_manager,
        device=DEVICE,
        val_split=VAL_SPLIT_PCT,     # <--- Passo lo split
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        p_aug=P
    )

    # 6. Start
    trainer.fit()

Configurazione Modello...
--- Inizializzazione Trainer (Val Split: 25.0%) ---
1. Caricamento dati da: ./drive/MyDrive/BBA_Dataset_training
   Split 75/25 -> Train: 126, Val: 42
--- Nessun checkpoint trovato. Inizio training da zero. ---
------------------------------------------------------------
Inizio Training Session: Epoca 1 -> 50
------------------------------------------------------------
Epoch [1/50]
   Train Loss: 0.0222
   Val Loss:   0.0046 | Val Accuracy: 100.00%
   >>> Record Loss! (inf -> 0.0046). Salvataggio checkpoint completo...
------------------------------------------------------------
Epoch [2/50]
   Train Loss: 0.0289
   Val Loss:   0.0061 | Val Accuracy: 100.00%
   >>> Nessun miglioramento (Best: 0.0046).
------------------------------------------------------------
Epoch [3/50]
   Train Loss: 0.0280
   Val Loss:   0.0006 | Val Accuracy: 100.00%
   >>> Record Loss! (0.0046 -> 0.0006). Salvataggio checkpoint completo...
----------------------------------------------